[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/255ribeiro/python_build123d_basics/blob/master/docs/tuto_colab_build/build123d_selecao_vertices_faces_gc.ipynb)

# Seleção de Vértices, Faces e Arestas
## build123d para Arquitetos e Engenheiros
### Versão Google Colab

---

Neste notebook vamos aprender a **selecionar vértices, faces e arestas** de um sólido e usar essas seleções para posicionar geometria em pontos específicos — a base para furos, encaixes, texturas e detalhamento paramétrico.

---

## Instalação

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "cadquery-simpleviewer[build123d]"],
        check=True,
    )
    # build123d pulls in a newer ipython than Colab's kernel bootstrap
    # tolerates. Put Colab's version back on disk — do NOT restart the
    # runtime, the current kernel already has the working ipython loaded.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "ipython==7.34.0", "--no-deps"],
        check=True,
    )

else:
    print("Not running in Colab, skipping package installation.")

## Importações

In [ ]:
import random

import build123d as b3d
from cadquery_simpleviewer import show

---

## Seletores de Topologia

Todo objeto do build123d guarda referências aos seus elementos de topologia: **vértices** (pontos), **arestas** (curvas) e **faces** (superfícies). Cada sólido oferece métodos para listá-los:

| Método | Retorna |
|--------|---------|
| `.vertices()` | Lista de `Vertex` — os pontos do sólido |
| `.edges()`    | Lista de `Edge` — as arestas (retas ou curvas) |
| `.faces()`    | Lista de `Face` — as superfícies |

Esses métodos retornam um `ShapeList` — uma lista "inteligente" que, além de indexável como uma lista comum, pode ser **ordenada** e **filtrada**.

---

In [ ]:
caixa = b3d.Box(4.0, 3.0, 2.0)

print(f"Vértices: {len(caixa.vertices())}")
print(f"Arestas:  {len(caixa.edges())}")
print(f"Faces:    {len(caixa.faces())}")

show([caixa], names=["Caixa"], visible_axes=None, z=-1.0, plane_size=6)

### Exemplo 1 — Esfera centrada em um vértice

Para posicionar uma forma em um vértice, extraímos suas coordenadas com `tuple(vertice)` e usamos `Pos()` para mover a nova geometria até lá:

```python
vertice = caixa.vertices()[0]
esfera = b3d.Pos(*tuple(vertice)) * b3d.Sphere(raio)
```

> 💡 O índice `[0]` pega apenas o primeiro vértice da lista, na ordem interna do kernel — não necessariamente previsível. Mais adiante veremos como escolher um vértice **específico** com `sort_by()`.

In [ ]:
caixa = b3d.Box(4.0, 3.0, 2.0)
raio = 0.8

vertice = caixa.vertices()[0]
esfera_vertice = b3d.Pos(*tuple(vertice)) * b3d.Sphere(raio)

show(
    [caixa, esfera_vertice],
    names=["Caixa", "Esfera no vértice"],
    colors=["lightsteelblue", "lightsalmon"],
    opacity=0.7,
    visible_axes=None,
    z=-1.0,
    plane_size=6
)

In [ ]:
resultado_vertice = caixa - esfera_vertice

show(
    [resultado_vertice],
    names=["Caixa - esfera no vértice"],
    colors=["lightsteelblue"],
    visible_axes=None,
    z=-1.0,
    plane_size=6
)

Subtração é só uma das operações booleanas disponíveis. As mesmas duas formas — `caixa` e `esfera_vertice` — podem ser combinadas de outras maneiras:

| Operação | Método | Operador | Resultado |
|----------|--------|----------|-----------|
| União | `.fuse(outro)` | `a + b` | Une os dois sólidos em um único volume |
| Subtração | `.cut(outro)` | `a - b` | Remove o volume do segundo sólido do primeiro |
| Interseção | `.intersect(outro)` | `a & b` | Mantém apenas o volume compartilhado pelos dois |

In [ ]:
# --- União ---
uniao_vertice = caixa + esfera_vertice

show(
    [uniao_vertice],
    names=["Caixa + esfera no vértice"],
    colors=["lightsteelblue"],
    visible_axes=None,
    z=-1.0,
    plane_size=6
)

In [ ]:
# --- Interseção ---
intersecao_vertice = caixa & esfera_vertice

show(
    [intersecao_vertice],
    names=["Caixa & esfera no vértice"],
    colors=["lightsalmon"],
    visible_axes=None,
    z=-1.0,
    plane_size=6
)

As três operações também podem ser **combinadas** entre si. Um exemplo clássico é a **diferença simétrica** (XOR): o que existe em um dos dois sólidos, mas não nos dois ao mesmo tempo — ou seja, a união menos a interseção:

```python
diferenca_simetrica = (a + b) - (a & b)
```

No canto da caixa isso isola justamente a "casca" onde a esfera cruza a aresta — o pedaço da esfera que fica para fora da caixa, mais o pedaço da caixa que fica dentro da esfera.

In [ ]:
# --- Combinação: diferença simétrica (XOR) ---
diferenca_simetrica = (caixa + esfera_vertice) - (caixa & esfera_vertice)

show(
    [diferenca_simetrica],
    names=["Diferença simétrica"],
    colors=["mediumpurple"],
    visible_axes=None,
    z=-1.0,
    plane_size=6
)

---

## Trocando o elemento selecionado

Escolher `[0]` é pouco previsível — o índice depende da ordem interna do kernel. Para selecionar um vértice, face ou aresta **específico**, combine `sort_by()`, `filter_by()`, `group_by()` e os operadores `>` / `<`:

| Técnica | Exemplo | Efeito |
|---------|---------|--------|
| `sort_by(eixo)[i]` | `.vertices().sort_by(Axis.Z)[-1]` | Ordena os elementos pela posição ao longo do eixo; `[-1]` pega o maior valor, `[0]` o menor |
| `> eixo` / `< eixo` | `.faces() > Axis.Z` | Atalho para `sort_by`: `>` ordena crescente, `<` decrescente |
| `filter_by(eixo)` | `.faces().filter_by(Axis.Z)` | Mantém só as faces cuja **normal** é paralela ao eixo (ou as arestas **paralelas** a ele) |
| `group_by(eixo)` | `.vertices().group_by(Axis.Z)` | Agrupa em sublistas por posição ao longo do eixo — útil para pegar "todos os vértices do topo" de uma vez |

> ⚠️ `filter_by(Axis.Z)` em faces devolve topo **e** base (as duas são perpendiculares a Z) — se quiser só uma delas, combine com `sort_by`/`>`/`<` e pegue o primeiro ou o último item.

In [ ]:
caixa = b3d.Box(4.0, 3.0, 2.0)

vertice_topo       = caixa.vertices().sort_by(b3d.Axis.Z)[-1]
face_topo          = (caixa.faces() > b3d.Axis.Z)[-1]
arestas_verticais  = caixa.edges().filter_by(b3d.Axis.Z)

print("Vértice do topo:            ", tuple(vertice_topo))
print("Centro da face do topo:     ", face_topo.center())
print("Arestas verticais na caixa: ", len(arestas_verticais))
print("Centro de uma aresta vertical:", arestas_verticais[0].center())

### Exemplo 2 — Esfera centrada no centro de uma face

Toda `Face` tem um método `.center()` que devolve o ponto médio da superfície — a forma mais direta de posicionar algo no meio de uma parede, laje ou fachada.

In [ ]:
caixa = b3d.Box(4.0, 3.0, 2.0)
raio = 0.8

face_frontal = caixa.faces().filter_by(b3d.Axis.Y)[0]
centro_face  = face_frontal.center()

esfera_face = b3d.Pos(centro_face) * b3d.Sphere(raio)

resultado_face = caixa - esfera_face

show(
    [resultado_face],
    names=["Caixa - esfera na face"],
    colors=["lightsteelblue"],
    visible_axes=None,
    z=-1.0,
    plane_size=6
)

Trocar a face selecionada não muda o resto do código — basta trocar o eixo em `filter_by()`, ou usar `>`/`<` para escolher entre as duas faces que ele encontrar:

In [ ]:
caixa = b3d.Box(4.0, 3.0, 2.0)
raio = 0.6

face_topo   = (caixa.faces() > b3d.Axis.Z)[-1]
esfera_topo = b3d.Pos(face_topo.center()) * b3d.Sphere(raio)

resultado_topo = caixa - esfera_topo

show(
    [resultado_topo],
    names=["Caixa - esfera no topo"],
    colors=["lightsteelblue"],
    visible_axes=None,
    z=-1.0,
    plane_size=6
)

### Exemplo 3 — Esfera centrada no meio de uma aresta

Arestas têm `.position_at(parametro)`. Com `parametro=0.5` — metade do percurso normalizado da aresta — obtemos o ponto médio, equivalente a `.center()` para arestas retas.

In [ ]:
caixa = b3d.Box(4.0, 3.0, 2.0)
raio = 0.5

aresta      = caixa.edges().filter_by(b3d.Axis.Z)[0]
meio_aresta = aresta.position_at(0.5)

esfera_aresta = b3d.Pos(meio_aresta) * b3d.Sphere(raio)

resultado_aresta = caixa - esfera_aresta

show(
    [resultado_aresta],
    names=["Caixa - esfera na aresta"],
    colors=["lightsteelblue"],
    visible_axes=None,
    z=-1.0,
    plane_size=6
)

---

## Pontos fora do centro

Além do centro, `Face` e `Edge` aceitam **coordenadas paramétricas** para qualquer ponto da superfície ou da curva:

- `face.position_at(u, v)` — `u` e `v` variam de `0` a `1` e percorrem a face nas duas direções da sua parametrização. Para uma face plana e retangular, como as de uma `Box`, correspondem diretamente às duas dimensões da face — `(0.5, 0.5)` é o centro, `(0, 0)` é um canto.
- `edge.position_at(posicao, position_mode)` — `posicao` também vai de `0` a `1` por padrão (`PositionMode.PARAMETER`). Com `position_mode=b3d.PositionMode.LENGTH`, `posicao` passa a ser interpretada como uma **distância real** ao longo da aresta, não mais uma fração.

> ⚠️ Em faces curvas (cilindros, esferas...) a relação entre `(u, v)` e a distância real percorrida na superfície não é linear — pontos igualmente espaçados em `u, v` podem ficar mais próximos ou mais afastados fisicamente. Em faces planas, como as de uma `Box`, a relação é direta.

In [ ]:
caixa = b3d.Box(4.0, 3.0, 2.0)
raio = 0.4

face = caixa.faces().filter_by(b3d.Axis.Y)[0]
ponto_face = face.position_at(0.2, 0.8)   # canto da face, não o centro
esfera_ponto_face = b3d.Pos(ponto_face) * b3d.Sphere(raio)

aresta = caixa.edges().filter_by(b3d.Axis.X)[0]
ponto_aresta = aresta.position_at(1.0, b3d.PositionMode.LENGTH)  # 1 unidade a partir do início da aresta
esfera_ponto_aresta = b3d.Pos(ponto_aresta) * b3d.Sphere(raio)

resultado_pontos = caixa - esfera_ponto_face - esfera_ponto_aresta

show(
    [resultado_pontos],
    names=["Caixa - pontos específicos"],
    colors=["lightsteelblue"],
    visible_axes=None,
    z=-1.0,
    plane_size=6
)

---

## Textura aleatória — pontos aleatórios em uma face

Combinando `position_at(u, v)` com `random.uniform(0, 1)` conseguimos gerar **furos em posições aleatórias** sobre uma face — útil para simular texturas, chapas microperfuradas ou acabamentos irregulares.

```python
u, v = random.uniform(0, 1), random.uniform(0, 1)
ponto = face.position_at(u, v)
```

Para subtrair muitas esferas de uma vez, é mais eficiente **somá-las primeiro** (`+`) em um único objeto e fazer **um só corte** no final, em vez de encadear várias subtrações:

> 💡 A face de uma `Box` é plana e retangular, então `u` e `v` distribuem os pontos de forma uniforme sobre toda a área. Usamos `random.seed()` apenas para que o resultado seja reproduzível neste notebook — remova essa linha para gerar um padrão diferente a cada execução.

In [ ]:
random.seed(42)

caixa      = b3d.Box(6.0, 4.0, 0.3)
raio_furo  = 0.15
n_furos    = 40
margem     = 0.05   # evita furos rente à borda da face

face_furada = (caixa.faces() > b3d.Axis.Z)[-1]

furos_total = None
for _ in range(n_furos):
    u = random.uniform(margem, 1 - margem)
    v = random.uniform(margem, 1 - margem)
    ponto = face_furada.position_at(u, v)
    furo = b3d.Pos(ponto) * b3d.Sphere(raio_furo)

    if furos_total is None:
        furos_total = furo
    else:
        furos_total += furo

chapa_perfurada = caixa - furos_total

show(
    [chapa_perfurada],
    names=["Chapa com perfuração aleatória"],
    colors=["lightsteelblue"],
    visible_axes=None,
    z=-0.5,
    plane_size=8
)

---

## Exercício

Crie um **painel decorativo perfurado** combinando os seletores aprendidos neste notebook:

1. Crie uma `Box` representando um painel (ex.: `5.0 x 3.0 x 0.2`)
2. Selecione a face frontal com `filter_by()` ou com o operador `>`/`<`
3. Gere pelo menos 30 pontos aleatórios sobre essa face com `position_at(u, v)`
4. Crie uma esfera em cada ponto, some todas em um único objeto e subtraia-o do painel de uma só vez
5. Exiba o resultado com `show()`

Dica: experimente `random.seed()` e raios diferentes para variar o padrão da perfuração, e compare o resultado de furar a face frontal com o de furar a face do topo.

In [ ]:
# Escreva seu código aqui

---

## Resumo

Neste notebook você aprendeu:

- Como listar a topologia de um sólido com `.vertices()`, `.edges()` e `.faces()`, que retornam um `ShapeList`
- Como escolher um elemento **específico** com `sort_by()`, `filter_by()`, `group_by()` e os operadores `>` / `<`
- Como posicionar uma esfera **em um vértice** usando `tuple(vertice)` e `Pos()`
- Como combinar duas formas com as três operações booleanas — união (`+`), subtração (`-`) e interseção (`&`) — e como compor combinações como a diferença simétrica (`(a + b) - (a & b)`)
- Como posicionar uma esfera **no centro de uma face** com `face.center()`
- Como posicionar uma esfera **no meio de uma aresta** com `edge.position_at(0.5)`
- Como obter pontos **fora do centro** com `face.position_at(u, v)` e `edge.position_at(distancia, PositionMode.LENGTH)`
- Como gerar **pontos aleatórios** sobre uma face com `random.uniform()`, somar várias formas em um único objeto e subtraí-lo de uma só vez

---
*build123d para Arquitetos e Engenheiros — Versão Google Colab*